In [ ]:
import os, time, math, json
from datetime import datetime, timedelta, timezone
import sys
sys.path.append("../")
import random
from qdrant_client import QdrantClient
import numpy as np
import polars as pl
from common.db import Connection
import polars as pl
from boto3.dynamodb.conditions import Attr



conn = Connection()
d = conn.get_dynamo()
dynamo = d.Table("event")

qdrant_client = QdrantClient(
    url=os.getenv("QDRANT_ENDPOINT"),
    api_key=os.getenv("QDRANT_API")
)

In [2]:
POS_EVENTS = {"article_in", "like", "archive", "share"}
IMP_EVENTS = {"f_imp", "b_imp"}

In [5]:
def now_utc_ms():
    return int(time.time() * 1000)

def fetch_events_last_7d(table_name: str = "event", region: str = "ap-northeast-2") -> pl.DataFrame:
    end_ms = now_utc_ms()
    start_ms = end_ms - 7 * 24 * 60 * 60 * 1000

    fe = Attr("timestamp").between(start_ms, end_ms)

    items = []
    last = None
    while True:
        params = {
            "FilterExpression": fe,
            "ProjectionExpression": "member_id, event_type, target_type, target_id, #ts",
            "ExpressionAttributeNames": {"#ts": "timestamp"},
            "Limit": 1000,
        }
        if last:  # None이면 넣지 않음 (중요)
            params["ExclusiveStartKey"] = last

        resp = dynamo.scan(**params)
        items.extend(resp.get("Items", []))
        last = resp.get("LastEvaluatedKey")
        if not last:
            break

    if not items:
        return pl.DataFrame(
            schema={
                "member_id": pl.Int64,
                "event_type": pl.Utf8,
                "target_type": pl.Utf8,
                "target_id": pl.Utf8,
                "timestamp": pl.Int64,
            }
        )

    # 정규화: 타입 강제
    norm = []
    for it in items:
        norm.append({
            "member_id": int(it["member_id"]),
            "event_type": str(it.get("event_type", "")),
            "target_type": str(it.get("target_type", "article")),
            "target_id": str(it.get("target_id", "")),
            "timestamp": int(it["timestamp"]),
        })

    return pl.DataFrame(norm)

In [6]:
df = fetch_events_last_7d()

In [7]:
df

member_id,event_type,target_type,target_id,timestamp
i64,str,str,str,i64
184,"""f_imp""","""article""","""5eAcUT3huFfSeDIaLNsEk1aUaAu""",1755261379796
184,"""f_imp""","""article""","""5eAcUT3huFfSeDIaLNsEk1aUaAu""",1755261380395
184,"""f_imp""","""article""","""c9MIUl1Gf7VMYjbWOlaRkf9P9nK""",1755261381407
184,"""f_imp""","""article""","""c9MIUl1Gf7VMYjbWOlaRkf9P9nK""",1755261382256
184,"""f_imp""","""article""","""hu2bqF1YlULJByQ8rv49nZRC5El""",1755261383115
…,…,…,…,…
182,"""f_imp""","""article""","""8K9wnURboXGrG73rQCt9sZFsNvm""",1755253220479
182,"""like""","""article""","""8K9wnURboXGrG73rQCt9sZFsNvm""",1755253228583
182,"""share""","""article""","""8K9wnURboXGrG73rQCt9sZFsNvm""",1755253232135
